# Chapter 4 — Citation Precision & Recall by Question Type and Category

Generates two grouped bar charts (AIT navy/teal theme) matching the other Chapter 4 figures:

- `fig_type_breakdown.png` — by the 6 question types
- `fig_category_breakdown.png` — by the 11 categories

Values are citation **precision** and **recall** (means over questions with gold sections),
computed by joining the real `question_type` from `pdpa_qa_dataset_clean.xlsx` with the
per-question citation metrics in `phase_4_Comparative_Analysis_grag_vrag_v2_0.xlsx`.
They are hard-coded below so the notebook runs standalone; run the optional last cell to
recompute them from the workbooks.

**Requirements:** `pip install matplotlib` (and `openpyxl` only for the optional recompute cell).

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs("figures", exist_ok=True)

# ---- AIT theme ----
NAVY = "#16233D"; TEAL = "#0E8C8C"; GREY = "#6B7280"; LTGREY = "#E5E7EB"
plt.rcParams.update({
    "font.family": "DejaVu Serif", "font.size": 11,
    "axes.edgecolor": GREY, "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": LTGREY, "grid.linewidth": 0.7,
    "axes.axisbelow": True, "figure.dpi": 200,
})

In [ ]:
# ---- Data: (n, VRAG_precision, VRAG_recall, KGRAG_precision, KGRAG_recall) ----
# np.nan marks an undefined value (no bar drawn).
TYPE = {
    "Definition":  (7,  0.786, 0.857, 0.667, 0.500),
    "Yes\u2013no": (11, 0.235, 0.636, 0.722, 0.545),
    "Explanatory": (5,  0.250, 0.375, 0.667, 0.500),
    "List":        (5,  0.367, 0.367, 0.800, 0.567),
    "Scenario":    (4,  0.125, 0.250, 0.500, 0.500),
    "Procedural":  (3,  0.000, 0.000, 0.250, 0.333),
}
CATEGORY = {
    "Compliance\nDocuments":     (7, 0.321, 0.643, 0.667, 0.571),
    "Consent":                   (5, 0.367, 0.433, 0.700, 0.600),
    "Definitions\n& Scope":      (4, 0.500, 0.750, 0.500, 0.375),
    "Lawful Basis":              (4, 0.208, 0.292, 1.000, 0.458),
    "Operational /\nProcedural": (4, 0.000, 0.000, 0.167, 0.250),
    "Roles & Actors":            (3, 0.667, 0.667, 1.000, 1.000),
    "Scope &\nExclusions":       (2, 0.167, 0.500, 1.000, 0.500),
    "Sensitive Data":            (2, 0.375, 0.750, 0.500, 0.500),
    "Governance\n& Authority":   (2, 0.333, 1.000, 0.000, 0.000),
    "Cross-border\nTransfer":    (1, 1.000, 1.000, 1.000, 1.000),
    "Data Subject\nRights":      (1, 0.000, 0.000, np.nan, 0.000),
}

In [ ]:
def grouped_chart(data, title, outfile, figsize, rot=0):
    labels = list(data.keys())
    ns = [data[k][0] for k in labels]
    vp = [data[k][1] for k in labels]; vr = [data[k][2] for k in labels]
    kp = [data[k][3] for k in labels]; kr = [data[k][4] for k in labels]
    x = np.arange(len(labels)); w = 0.20
    fig, ax = plt.subplots(figsize=figsize)
    series = [
        (vp, -1.5, NAVY, 1.00, "VRAG precision"),
        (vr, -0.5, NAVY, 0.45, "VRAG recall"),
        (kp,  0.5, TEAL, 1.00, "KGRAG precision"),
        (kr,  1.5, TEAL, 0.45, "KGRAG recall"),
    ]
    for vals, off, color, alpha, lab in series:
        bars = ax.bar(x + off*w, vals, w, color=color, alpha=alpha,
                      edgecolor=color, linewidth=0.6, label=lab)
        for r, v in zip(bars, vals):
            if np.isnan(v):
                ax.text(r.get_x()+r.get_width()/2, 0.02, "n/a", ha="center",
                        va="bottom", fontsize=6.5, color=GREY, rotation=90)
            elif v > 0:
                ax.text(r.get_x()+r.get_width()/2, v+0.015, f"{v:.2f}",
                        ha="center", va="bottom", fontsize=6, color=NAVY)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{l}\n(n={n})" for l, n in zip(labels, ns)],
                       fontsize=8.5, rotation=rot)
    ax.set_ylabel("Score"); ax.set_ylim(0, 1.08)
    ax.set_title(title, fontsize=11, color=NAVY, pad=10)
    ax.legend(frameon=False, ncol=4, fontsize=8.5,
              loc="upper center", bbox_to_anchor=(0.5, -0.14))
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.tick_params(length=0)
    plt.tight_layout(); plt.savefig(outfile, bbox_inches="tight")
    plt.show()
    print("saved", outfile)

In [ ]:
grouped_chart(TYPE, "Citation Precision and Recall by Question Type",
              "figures/fig_type_breakdown.png", (9.5, 5.2))

In [ ]:
grouped_chart(CATEGORY, "Citation Precision and Recall by Question Category",
              "figures/fig_category_breakdown.png", (13.5, 5.6))

## LaTeX include snippets

```latex
\begin{figure}[H]
  \centering
  \includegraphics[width=\linewidth]{figures/fig_type_breakdown.png}
  \caption{Citation precision and recall by question type.}
  \label{fig:type-breakdown}
\end{figure}

\begin{figure}[H]
  \centering
  \includegraphics[width=\linewidth]{figures/fig_category_breakdown.png}
  \caption{Citation precision and recall by question category.}
  \label{fig:category-breakdown}
\end{figure}
```

## Optional: recompute the values from the workbooks

Run this to regenerate the `TYPE` and `CATEGORY` dictionaries directly from your files,
instead of using the hard-coded values above. Adjust the two paths first.
It joins the **real** `question_type` from the source dataset (the Phase 4 workbook's
`question_type` column mistakenly holds category values) with the per-question metrics.

In [ ]:
# import openpyxl, statistics as st
# from collections import defaultdict
#
# SRC = "pdpa_qa_dataset_clean.xlsx"                       # source dataset
# WB  = "phase_4_Comparative_Analysis_grag_vrag_v2_0.xlsx" # metrics workbook
#
# src = openpyxl.load_workbook(SRC, data_only=True)["Dataset"]
# sh = [c.value for c in src[1]]
# idi, qti, ci = sh.index("id"), sh.index("question_type"), sh.index("category")
# TYPE_OF, CAT_OF = {}, {}
# for r in src.iter_rows(min_row=2, values_only=True):
#     if r[idi]: TYPE_OF[r[idi]] = r[qti]; CAT_OF[r[idi]] = r[ci]
#
# wb = openpyxl.load_workbook(WB, data_only=True)
# def rows(ws):
#     h = [c.value for c in ws[1]]
#     for r in ws.iter_rows(min_row=2, values_only=True):
#         if r[0] is not None: yield dict(zip(h, r))
# V = {r["id"]: r for r in rows(wb["citation recall - vrag"])}
# G = {r["id"]: r for r in rows(wb["citation recall - grag"])}
#
# def num(x): return x if isinstance(x, (int, float)) else None
# def mean(xs): return round(st.mean(xs), 3) if xs else float("nan")
#
# def build(mapping):
#     b = defaultdict(lambda: {"n":0,"vp":[],"vr":[],"kp":[],"kr":[]})
#     for i in V:
#         d = b[mapping[i]]; d["n"] += 1
#         if num(V[i].get("section_precision")) is not None: d["vp"].append(V[i]["section_precision"])
#         if num(V[i].get("section_recall"))    is not None: d["vr"].append(V[i]["section_recall"])
#         g = G.get(i, {})
#         if num(g.get("citation_precision")) is not None: d["kp"].append(g["citation_precision"])
#         if num(g.get("citation_recall"))    is not None: d["kr"].append(g["citation_recall"])
#     return {k: (d["n"], mean(d["vp"]), mean(d["vr"]), mean(d["kp"]), mean(d["kr"]))
#             for k, d in b.items()}
#
# TYPE = build(TYPE_OF); CATEGORY = build(CAT_OF)
# print(TYPE); print(CATEGORY)